### *ETAPA 6 Preparación del modelo analítico para Power BI*

En esta etapa se consolidan los resultados obtenidos durante el análisis en un único archivo Excel estructurado para su consumo desde Power BI. El objetivo es centralizar la información necesaria para construir el tablero ejecutivo solicitado por el cliente.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
# ==========================================
# Cargar dataset
# ==========================================

segmentacion = pd.read_excel(
    "../data/processed/segmentacion_final.xlsx"
)

transacciones = pd.read_excel(
    "../data/raw/DataTransacciones_PruebasIngreso.xlsx",
    sheet_name="transacciones"
)

In [6]:
# ==========================================
# Construcción KPIs
# ==========================================

ventas_totales = segmentacion["Monto ventas Acumulado"].sum()

comision_total = ventas_totales * 0.05

ventas_ultimo_mes = segmentacion["Monto ventas ultimo mes"].sum()

comision_ultimo_mes = ventas_ultimo_mes * 0.05

comercios_activos = (
    segmentacion["Monto ventas ultimo mes"] > 0
).sum()

ticket_promedio = (
    ventas_totales /
    segmentacion["frecuencia"].sum()
)

frecuencia_promedio = (
    segmentacion["frecuencia"].mean()
)

In [7]:
# ==========================================
# Tabla resumen KPIs
# ==========================================

kpis_generales = pd.DataFrame({

    "KPI":[

        "Ventas Totales",
        "Comisión Estimada",
        "Ventas Último Mes",
        "Comisión Último Mes",
        "Comercios Activos",
        "Ticket Promedio",
        "Frecuencia Promedio"

    ],

    "Valor":[

        ventas_totales,
        comision_total,
        ventas_ultimo_mes,
        comision_ultimo_mes,
        comercios_activos,
        ticket_promedio,
        frecuencia_promedio

    ]

})

In [8]:
# ==========================================
# KPIs por segmento
# ==========================================

dashboard_segmentos = (

    segmentacion

    .groupby("segmento")

    .agg(

        comercios=("Id_comercio","count"),

        transacciones=("frecuencia","sum"),

        ventas_acumuladas=("Monto ventas Acumulado","sum"),

        ventas_ultimo_mes=("Monto ventas ultimo mes","sum")

    )

)

In [9]:
dashboard_segmentos["comision_acumulada"] = (

    dashboard_segmentos["ventas_acumuladas"]*0.05

)

dashboard_segmentos["comision_ultimo_mes"]=(

    dashboard_segmentos["ventas_ultimo_mes"]*0.05

)

dashboard_segmentos["% ventas"]=(

    dashboard_segmentos["ventas_acumuladas"]

    /

    dashboard_segmentos["ventas_acumuladas"].sum()

)*100

dashboard_segmentos["% comisión"]=(

    dashboard_segmentos["comision_acumulada"]

    /

    dashboard_segmentos["comision_acumulada"].sum()

)*100

In [13]:
# ==========================================
# Histórico mensual de métricas
# ==========================================

historico = (

    transacciones

    .groupby("mes-anio transaccion", as_index=False)

    .agg(

        transacciones=("cantidad transacciones", "sum"),

        comercios_activos=("Id_comercio", "nunique")

    )

)

# Convertir a fecha para que Power BI ordene correctamente
historico["mes-anio transaccion"] = pd.to_datetime(

    historico["mes-anio transaccion"],

    format="%m-%Y"

)

historico = historico.sort_values(

    "mes-anio transaccion"

)

# Promedio de transacciones por comercio activo
historico["promedio_transacciones"] = (

    historico["transacciones"]

    /

    historico["comercios_activos"]

).round(2)

display(historico)

,mes-anio transaccion,transacciones,comercios_activos,promedio_transacciones
1,2020-12-01,1408,198,7.11
0,2021-01-01,2234,353,6.33
2,2021-02-01,6134,887,6.92
3,2021-03-01,18205,2737,6.65
4,2021-04-01,17222,2095,8.22
5,2021-05-01,22949,2087,11.00


In [17]:
# ==========================================
# Tabla resumen para Power BI
# ==========================================

segmentos = (
    segmentacion
    .groupby("segmento")
    .agg(
        comercios=("Id_comercio", "count"),
        ventas=("Monto ventas Acumulado", "sum"),
        ventas_mes=("Monto ventas ultimo mes", "sum")
    )
    .reset_index()
)

segmentos["participacion_ventas"] = (
    segmentos["ventas"] /
    segmentos["ventas"].sum()
    * 100
)

segmentos["participacion_ventas_mes"] = (
    segmentos["ventas_mes"] /
    segmentos["ventas_mes"].sum()
    * 100
)

segmentos = segmentos.round(2)

In [18]:
# ==========================================
# Exportar archivo final para Power BI
# ==========================================

from pathlib import Path

Path("../data/dashboard").mkdir(
    parents=True,
    exist_ok=True
)

with pd.ExcelWriter(
    "../data/dashboard/dashboard_prueba.xlsx",
    engine="openpyxl"
) as writer:

    # ==========================
    # Hoja 1 - KPIs
    # ==========================
    kpis_generales.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    # ==========================
    # Hoja 2 - Segmentos
    # ==========================
    segmentos.to_excel(
    writer,
    sheet_name="Segmentos",
    index=False
    )

    # ==========================
    # Hoja 3 - Comercios
    # ==========================
    segmentacion.to_excel(
        writer,
        sheet_name="Comercios",
        index=False
    )

    # ==========================
    # Hoja 4 - Histórico
    # ==========================
    historico.to_excel(
        writer,
        sheet_name="Historico",
        index=False
    )

print("✅ dashboard_prueba.xlsx exportado correctamente.")

✅ dashboard_prueba.xlsx exportado correctamente.
